In [1]:
import pandas as pd

In [2]:
!ls ../dataset_kraken/


family_name_ids.csv  genus_name_ids.csv  species_name_ids.csv
family_seqs.csv      genus_seqs.csv	 species_seqs.csv


In [2]:
species_labels = pd.read_csv("../dataset_kraken/species_name_ids.csv")
genus_labels = pd.read_csv("../dataset_kraken/genus_name_ids.csv")

In [3]:
species_labels.head()

,species,id
0,Alphafusarivirus rutstroemiae,1
1,Alphafusarivirus macrophominae,2
2,Alphafusarivirus ioaponiae,3
3,Alphafusarivirus auriculariae,4
4,Alphafusarivirus penicilli,5


In [5]:
genus_labels.head()

,genus,id
0,Alphafusarivirus,1
1,Betafusarivirus,2
2,Gammafusarivirus,3
3,Betapartitivirus,4
4,Gammapartitivirus,5


In [6]:
import pandas as pd
from Bio import Entrez, SeqIO
from tqdm import tqdm
import time

# 2. CONFIGURATION
Entrez.email = "matheuslimaaa13@gmail.com"  # <--- REPLACE THIS

def fetch_ncbi_sequences(species_name, max_results=3):
    """
    Searches NCBI Nucleotide for a species and returns a list of sequences.
    """
    try:
        search_term = f"{species_name}[Organism] AND biomol_genomic[PROP]"
        search_handle = Entrez.esearch(db="nucleotide", term=search_term, retmax=max_results)
        search_results = Entrez.read(search_handle)
        search_handle.close()
        
        id_list = search_results['IdList']
        
        if not id_list:
            return []

        fetch_handle = Entrez.efetch(db="nucleotide", id=id_list, rettype="fasta", retmode="text")
        sequences = [str(record.seq) for record in SeqIO.parse(fetch_handle, "fasta")]
        fetch_handle.close()
        
        return sequences

    except Exception as e:
        print(f"Error fetching {species_name}: {e}")
        return []

# 3. MAIN PROCESSING
final_data = []

# Using tqdm for the progress bar
# We wrap species_labels.iterrows() (or just the index) with tqdm
print("Starting processing...")
for index, row in tqdm(species_labels.iterrows(), total=species_labels.shape[0], desc="Fetching Species"):
    
    species_name = row['species']
    species_id = row['id']
    
    # 1. Derive Genus
    derived_genus_name = species_name.split()[0]
    
    # 2. Check and Update Genus DataFrame
    # Check if genus exists in the genus_labels DataFrame
    genus_match = genus_labels[genus_labels['genus'] == derived_genus_name]
    
    if genus_match.empty:
        # Create a new unique ID (max id + 1, or 1 if empty)
        if genus_labels.empty:
            new_genus_id = 1
        else:
            new_genus_id = genus_labels['id'].max() + 1
            
        # Add new row to genus_labels
        new_row = pd.DataFrame([{'genus': derived_genus_name, 'id': new_genus_id}])
        genus_labels = pd.concat([genus_labels, new_row], ignore_index=True)
        
        genus_id = new_genus_id
        # Optional: Print a small message (tqdm.write works better with progress bars)
        tqdm.write(f"Added new genus: {derived_genus_name} (ID: {genus_id})")
    else:
        # Get existing ID
        genus_id = genus_match.iloc[0]['id']
    
    # 3. Fetch from NCBI
    sequences = fetch_ncbi_sequences(species_name)
    
    # 4. Append Data
    for seq in sequences:
        final_data.append({
            'sequence': seq,
            'species_id': species_id,
            'genus_id': genus_id
        })
    
    # Sleep to respect NCBI rate limits
    time.sleep(1)

# 4. OUTPUT RESULTS
final_df = pd.DataFrame(final_data)

print("\nProcessing Complete!")
print("Updated Genus Table:")
print(genus_labels)
print("\nFinal Sequence Table:")
print(final_df.head())

Starting processing...


Fetching Species:   0%|          | 34/13683 [01:19<8:48:13,  2.32s/it]

Added new genus: Nigrospora (ID: 2703)


Fetching Species:   0%|          | 35/13683 [01:22<8:47:02,  2.32s/it]

Added new genus: Sodiomyces (ID: 2704)


...

Added new genus: Leshenault (ID: 4125)


Fetching Species: 100%|█████████▉| 13650/13683 [9:18:07<01:16,  2.32s/it]

Added new genus: Egaro (ID: 4126)


Fetching Species: 100%|██████████| 13683/13683 [9:19:21<00:00,  2.45s/it]


Processing Complete!
Updated Genus Table:
                  genus    id
0      Alphafusarivirus     1
1       Betafusarivirus     2
2      Gammafusarivirus     3
3      Betapartitivirus     4
4     Gammapartitivirus     5
...                 ...   ...
4121           Planaria  4122
4122             Duwamo  4123
4123             Abisko  4124
4124         Leshenault  4125
4125              Egaro  4126

[4126 rows x 2 columns]

Final Sequence Table:
                                            sequence  species_id  genus_id
0  CCGGGGTGTTAAAGATAACGTTAATATGTGTTATCCAGTATGTCAT...           1         1
1  CCGGGGTGTTAAAGATAACGTTAATATGTGTTATCCAGTATGTCAT...           1         1
2  AATCCCAATCGCACCCACCCAGCACCGCTCTGAGTGCTATTAAGAA...           2         1
3  AATCCCAATCGCACCCACCCAGCACCGCTCTGAGTGCTATTAAGAA...           2         1
4  CTCACATAGAGAACCACGGGCTTCTTTCTTTTGCCCTTTTTCCACT...           3         1


In [5]:
genus_labels.to_csv("raw_database/genus_new_labels.csv")
final_df.to_csv("raw_database/final.csv")

In [6]:
df_labels_genus = pd.read_csv("raw_database/genus_new_labels.csv")
final_df = pd.read_csv("raw_database/final.csv")

In [7]:
df_labels_genus.columns

Index(['Unnamed: 0', 'genus', 'id'], dtype='object')

In [8]:
final_df.columns

Index(['Unnamed: 0', 'sequence', 'species_id', 'genus_id'], dtype='object')

In [ ]:
import pandas as pd
from Bio import Entrez, SeqIO
from tqdm import tqdm
import time

# 2. CONFIGURATION
Entrez.email = "matheuslimaaa13@gmail.com"  # <--- REPLACE THIS

def get_ncbi_taxonomy_genus(species_name):
    """
    Fetches the taxonomic lineage. 
    Priority 1: Find a node with Rank='genus'.
    Priority 2 (Viruses): If no genus, take the immediate parent (last item in lineage).
    """
    try:
        # Search for the species ID
        # Using [All Fields] helps find names that might be synonyms or isolates
        search_handle = Entrez.esearch(db="taxonomy", term=f"{species_name}", retmode="xml")
        search_results = Entrez.read(search_handle)
        search_handle.close()
        
        if not search_results['IdList']:
            return None, "Search Failed"
            
        # Use the first ID found
        tax_id = search_results['IdList'][0]
        
        # Fetch taxonomy details
        fetch_handle = Entrez.efetch(db="taxonomy", id=tax_id, retmode="xml")
        data = Entrez.read(fetch_handle)
        fetch_handle.close()
        
        if not data:
            return None, "No Data"

        record = data[0]
        lineage = record.get('LineageEx', [])
        
        # 1. Try to find explicit Genus rank
        for tax_node in lineage:
            if tax_node['Rank'] == 'genus':
                return tax_node['ScientificName'], "Found Genus"
        
        # 2. Fallback: If it's a Virus and no Genus found, take the immediate parent
        # Check if 'Viruses' is in the lineage to confirm it's viral
        is_virus = any(node['ScientificName'] == 'Viruses' for node in lineage)
        
        if is_virus and lineage:
            # The last item in LineageEx is the direct parent
            parent_name = lineage[-1]['ScientificName']
            return parent_name, f"Fallback to Parent ({lineage[-1]['Rank']})"
            
        return None, "No Genus or Parent found"

    except Exception as e:
        return None, f"Error: {str(e)}"

# 3. PROCESSING
unique_species_ids = final_df['species_id'].unique()
print(f"Processing {len(unique_species_ids)} unique species...")

for sp_id in tqdm(unique_species_ids, desc="Repairing Lineage"):
    
    # Get the Species Name
    sp_row = species_labels[species_labels['id'] == sp_id]
    if sp_row.empty:
        continue
    species_name = sp_row.iloc[0]['species']
    
    # Get True Genus from NCBI
    true_genus_name, status = get_ncbi_taxonomy_genus(species_name)
    
    if not true_genus_name:
        # If it fails, we keep the original (or you can mark it 'Unknown')
        # tqdm.write(f"FAILED: {species_name} -> {status}")
        continue
    
    # Update/Find Genus in df_labels_genus
    genus_match = df_labels_genus[df_labels_genus['genus'] == true_genus_name]
    
    if genus_match.empty:
        # Create new ID
        new_id = df_labels_genus['id'].max() + 1 if not df_labels_genus.empty else 1
        new_row = pd.DataFrame([{'genus': true_genus_name, 'id': new_id}])
        df_labels_genus = pd.concat([df_labels_genus, new_row], ignore_index=True)
        correct_genus_id = new_id
    else:
        correct_genus_id = genus_match.iloc[0]['id']
        
    # Fix final_df
    final_df.loc[final_df['species_id'] == sp_id, 'genus_id'] = correct_genus_id
    
    # Optional: Print updates for the previously failing ones to verify
    if "Fallback" in status:
        tqdm.write(f"Fixed: {species_name} -> {true_genus_name} ({status})")
    
    time.sleep(0.5) # Rate limit

# 4. CLEANUP (Remove unused genus labels)
print("\nCleaning up unused Genus labels...")
valid_genus_ids = final_df['genus_id'].unique()
df_labels_genus = df_labels_genus[df_labels_genus['id'].isin(valid_genus_ids)].copy()
df_labels_genus.reset_index(drop=True, inplace=True)

# 5. SAVE OR VIEW
print("Repair Complete.")
print(df_labels_genus.head())
print(final_df.head())

Processing 13592 unique species...


Repairing Lineage:   0%|          | 34/13592 [00:57<5:57:52,  1.58s/it]

Fixed: Nigrospora oryzae fusarivirus 1 -> unclassified Fusariviridae (Fallback to Parent (no rank))


Repairing Lineage:   0%|          | 35/13592 [00:59<6:26:46,  1.71s/it]

Fixed: Sodiomyces alkalinus fusarivirus 1 -> unclassified Fusariviridae (Fallback to Parent (no rank))


...

Repairing Lineage:  33%|███▎      | 4442/13592 [1:57:00<3:57:28,  1.56s/it]

Fixed: Drosophila melanogaster birnavirus SW-2009a -> unclassified Birnaviridae (Fallback to Parent (no rank))


Repairing Lineage:  33%|███▎      | 4443/13592 [1:57:02<3:54:17,  1.54s/it]

Fixed: Espirito Santo virus -> unclassified Birnaviridae (Fallback to Parent (no rank))


Repairing Lineage:  33%|███▎      | 4495/13592 [1:58:22<3:52:50,  1.54s/it]

In [13]:
df_labels_genus.to_csv("raw_database/genus_new_labels.csv")
final_df.to_csv("raw_database/final.csv")

In [ ]:
!touch raw_database/fim.txt